# 🦆 DuckDB Explorer — Immigration Database
**Project-00: US Immigration Data Platform**

Use this notebook to explore, query, and sanity-check the immigration database.
Run cells top to bottom on first use.

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────────────────
import duckdb
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.0f}'.format)
pd.set_option('display.width', 200)

# Connect to the database
DB_PATH = '../database/immigration.duckdb'
con = duckdb.connect(DB_PATH, read_only=True)
print('✅ Connected to:', DB_PATH)

In [ ]:
# ── WHAT TABLES EXIST? ─────────────────────────────────────────────────────
print('📋 Tables in database:')
con.execute("SHOW TABLES").df()

In [ ]:
# ── WHAT COLUMNS EXIST? ────────────────────────────────────────────────────
# Change table name if you add more tables later
TABLE = 'visa_issuances'

print(f'📊 Schema for {TABLE}:')
con.execute(f"DESCRIBE {TABLE}").df()

In [ ]:
# ── QUICK STATS ────────────────────────────────────────────────────────────
stats = con.execute("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT country) as countries,
        COUNT(DISTINCT fiscal_year) as fiscal_years,
        MIN(fiscal_year) as earliest_year,
        MAX(fiscal_year) as latest_year
    FROM visa_issuances
""").df()
print('📈 Database Stats:')
stats

In [ ]:
# ── ALL COLUMN NAMES (visa types) ──────────────────────────────────────────
cols = con.execute("SELECT column_name FROM information_schema.columns WHERE table_name = 'visa_issuances' ORDER BY ordinal_position").df()
print(f'Total columns: {len(cols)}')
print('\nAll columns:')
for i, col in enumerate(cols['column_name'].tolist()):
    print(f'  {i:3d}. {col}')

In [ ]:
# ── PREVIEW RAW DATA ───────────────────────────────────────────────────────
# Change LIMIT to see more rows
con.execute("""
    SELECT fiscal_year, country, "H-1B", "F-1", "B-1/B-2"
    FROM visa_issuances
    ORDER BY fiscal_year DESC, "H-1B" DESC
    LIMIT 20
""").df()

---
## 🔍 Custom SQL Query Cell
Write any SQL query below and run it.

In [ ]:
# ── YOUR CUSTOM QUERY ──────────────────────────────────────────────────────
# Edit the SQL below and run this cell

query = """
SELECT 
    country,
    fiscal_year,
    "H-1B"
FROM visa_issuances
WHERE country IN ('India', 'China - mainland', 'Mexico')
  AND fiscal_year >= 2015
ORDER BY fiscal_year DESC, "H-1B" DESC
"""

result = con.execute(query).df()
print(f'Rows returned: {len(result)}')
result

---
## 📊 Quick Analysis Queries
Pre-built queries for common questions. Run any cell below.

In [ ]:
# TOP 10 COUNTRIES BY H-1B — any year
YEAR = 2024  # ← change this

con.execute(f"""
    SELECT country, "H-1B" as h1b_visas
    FROM visa_issuances
    WHERE fiscal_year = {YEAR}
    ORDER BY "H-1B" DESC
    LIMIT 10
""").df()

In [ ]:
# H-1B TREND FOR ANY COUNTRY over time
COUNTRY = 'India'  # ← change this

con.execute(f"""
    SELECT fiscal_year, "H-1B" as h1b_visas
    FROM visa_issuances
    WHERE country = '{COUNTRY}'
    ORDER BY fiscal_year ASC
""").df()

In [ ]:
# TOTAL VISAS BY COUNTRY — FY2024 all types combined
# Gets all numeric columns and sums them
con.execute("""
    SELECT 
        country,
        fiscal_year,
        "Grand Total" as total_visas
    FROM visa_issuances
    WHERE fiscal_year = 2024
    ORDER BY "Grand Total" DESC
    LIMIT 20
""").df()

In [ ]:
# COMPARE TWO COUNTRIES across all years for any visa type
COUNTRY_1 = 'India'
COUNTRY_2 = 'China - mainland'
VISA_TYPE = 'H-1B'  # ← change to any visa type from the column list above

con.execute(f"""
    SELECT 
        fiscal_year,
        MAX(CASE WHEN country = '{COUNTRY_1}' THEN "{VISA_TYPE}" END) as "{COUNTRY_1}",
        MAX(CASE WHEN country = '{COUNTRY_2}' THEN "{VISA_TYPE}" END) as "{COUNTRY_2}"
    FROM visa_issuances
    WHERE country IN ('{COUNTRY_1}', '{COUNTRY_2}')
    GROUP BY fiscal_year
    ORDER BY fiscal_year DESC
""").df()

In [ ]:
# FIND ANY COUNTRY NAME (if you're not sure of exact spelling)
SEARCH = 'china'  # ← lowercase search term

con.execute(f"""
    SELECT DISTINCT country 
    FROM visa_issuances 
    WHERE LOWER(country) LIKE '%{SEARCH}%'
    ORDER BY country
""").df()

In [ ]:
# CLOSE CONNECTION when done
con.close()
print('✅ Connection closed cleanly.')